# Tutorial 1: Learn about `requests`

Follow along with [this tutorial on RealPython](https://realpython.com/python-requests/). 
Complete the first 5 sections thoroughly (up to but not including User Other HTTP methods). Then jump to section Improve Performance to learn about some advanced tricks that you might need at some point when scraping websites that have a lot of content.

Use markdown headings as appropriate to enumerate sections.

#### Author: Kelly Chen
#### Date: 9/9/26

Import the Requests library; if not installed, use python -m pip install requests

In [1]:
import requests

## 1. Make a GET request

HTTP methods, such as GET and POST, specify the action you want to perform when making an HTTP request.

In [3]:
# make GET request to GitHub's REST API
requests.get("https://api.github.com")

<Response [200]>

## 2. Inspect the Response


In [6]:
response = requests.get("https://api.github.com")
response.status_code

200

Response status code of 200 means that your request was successful and the server responded with the data you requested.

In [7]:
if response.status_code == 200:
    print("Success!")
elif response.status_code == 404:
    print("Not Found.")

Success!


In [8]:
if response:
    print("Success!")
else:
    raise Exception(f"Non-success status code: {response.status_code}")

Success!


This code implicitly checks whether the .status_code of response is between 200 and 399. If it’s not, then you raise an exception with an error message that includes the non-success status code wrapped in an f-string.

**Additionally:** use Request’s built-in capacities to raise an exception if the request was unsuccessful. You can do this using .raise_for_status():

In [9]:
import requests
from requests.exceptions import HTTPError

URLS = ["https://api.github.com", "https://api.github.com/invalid"]

for url in URLS:
    try:
        response = requests.get(url)
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        print("Success!")

Success!
HTTP error occurred: 404 Client Error: Not Found for url: https://api.github.com/invalid


#### Access the Response Content


The response of a GET request contains valuable information, known as the payload, in the message body.

In [11]:
# use .content to see response contents in bytes
response = requests.get("https://api.github.com")
response.content

type(response.content)

bytes

You’ll often want to convert them into a string using a character encoding such as UTF-8. response will do that for you when you access .text:

In [12]:
response.text

type(response.text)

str

You can provide an explicit encoding by setting .encoding before accessing .text:

In [13]:
response.encoding = "utf-8"  # Optional: Requests infers this.
response.text

'{\n  "current_user_url": "https://api.github.com/user",\n  "current_user_authorizations_html_url": "https://github.com/settings/connections/applications{/client_id}",\n  "authorizations_url": "https://api.github.com/authorizations",\n  "code_search_url": "https://api.github.com/search/code?q={query}{&page,per_page,sort,order}",\n  "commit_search_url": "https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}",\n  "emails_url": "https://api.github.com/user/emails",\n  "emojis_url": "https://api.github.com/emojis",\n  "events_url": "https://api.github.com/events",\n  "feeds_url": "https://api.github.com/feeds",\n  "followers_url": "https://api.github.com/user/followers",\n  "following_url": "https://api.github.com/user/following{/target}",\n  "gists_url": "https://api.github.com/gists{/gist_id}",\n  "hub_url": "https://api.github.com/hub",\n  "issue_search_url": "https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}",\n  "issues_url": "https://api.g

In [15]:
# use json.loads() to deserialize and JSON content
response.json()

type(response.json())

dict

In [16]:
response_dict = response.json()
response_dict["emojis_url"]

'https://api.github.com/emojis'

#### View Response Headers

Headers give informations like content type of response payload and how long it takes to cache.

In [17]:
# view headers
import requests

response = requests.get("https://api.github.com")
response.headers

{'Date': 'Wed, 09 Sep 2026 17:36:39 GMT', 'Cache-Control': 'public, max-age=60, s-maxage=60', 'Vary': 'Accept,Accept-Encoding, Accept, X-Requested-With', 'ETag': '"4f825cc84e1c733059d46e76e6df9db557ae5254f9625dfe8e1b09499c449438"', 'x-github-api-version-selected': '2022-11-28', 'Access-Control-Expose-Headers': 'ETag, Link, Location, Retry-After, X-GitHub-OTP, X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Used, X-RateLimit-Resource, X-RateLimit-Reset, X-OAuth-Scopes, X-Accepted-OAuth-Scopes, X-Poll-Interval, X-GitHub-Media-Type, X-GitHub-SSO, X-GitHub-Request-Id, Deprecation, Sunset, Warning', 'Access-Control-Allow-Origin': '*', 'Strict-Transport-Security': 'max-age=31536000; includeSubdomains; preload', 'X-Frame-Options': 'deny', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Referrer-Policy': 'origin-when-cross-origin, strict-origin-when-cross-origin', 'Content-Security-Policy': "default-src 'none'", 'Server': 'github.com', 'Content-Type': 'application/json; ch

In [19]:
# .headers returns dict-like object to access header values by key
response.headers["Content-Type"]

'application/json; charset=utf-8'

## 3. Add Query String Parameters

One common way to customize a GET request is to pass values through query string parameters in the URL. To do this using get(), you pass data to params. 

In [20]:
import requests

response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": "language:python", "sort": "stars", "order": "desc"},
)

json_response = response.json()
popular_repositories = json_response["items"]
for repo in popular_repositories[:3]:
    print(f"Name: {repo['name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}\n")

Name: public-apis
Description: A collective list of free APIs
Stars: 478637

Name: free-programming-books
Description: :books: Freely available programming books
Stars: 396457

Name: system-design-primer
Description: Learn how to design large-scale systems. Prep for the system design interview.  Includes Anki flashcards.
Stars: 369288



In [22]:
# list of tuples

requests.get(
    "https://api.github.com/search/repositories",
    [("q", "language:python"), ("sort", "stars"), ("order", "desc")],
)

<Response [200]>

In [23]:
# pass as bytes
requests.get(
    "https://api.github.com/search/repositories",
    params=b"q=language:python&sort=stars&order=desc",
)

<Response [200]>

## 4. Customize Request Headers

To customize headers, you pass a dictionary of HTTP headers to get() using the headers parameter. 

In [ ]:
# The Accept header tells the server what content types your application can handle
response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": '"real python"'},
    headers={"Accept": "application/vnd.github.text-match+json"}, 
)

json_response = response.json()
first_repository = json_response["items"][0]
print(first_repository["text_matches"][0]["matches"])

[{'text': 'Real Python', 'indices': [23, 34]}]


## 5. Improve Performance

#### Set Request Timeouts
By default, Requests will wait indefinitely on the response, so you should almost always specify a timeout duration to prevent these issues from happening. To set the request’s timeout, use the timeout parameter. timeout can be an integer or float representing the number of seconds to wait on a response before timing out:

In [ ]:
requests.get("https://api.github.com", timeout=1) #successful

<Response [200]>

In [30]:
requests.get("https://api.github.com", timeout=0.01) #unsuccessful; timed out

ConnectTimeout: HTTPSConnectionPool(host='api.github.com', port=443): Max retries exceeded with url: / (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x1201f16d0>, 'Connection to api.github.com timed out. (connect timeout=0.01)'))

You can also pass a tuple to timeout with the following two elements:

- Connect timeout: The amount of time it allows the client to establish a connection to the server
- Read timeout: The time it’ll wait for a response once the client has established a connection


In [31]:
requests.get("https://api.github.com", timeout=(3.05, 5))

<Response [200]>

In [32]:
import requests
from requests.exceptions import Timeout

try:
    response = requests.get("https://api.github.com", timeout=(3.05, 5))
except Timeout:
    print("The request timed out")
else:
    print("The request did not time out")

The request did not time out


#### Reuse Connections With Session Objects


If you need to fine-tune your control over how requests are being made or improve the performance of your requests, you may need to use a Session instance directly. Sessions are used to persist parameters across requests. For example, if you want to use the same authentication across multiple requests, then you can use a session:


In [ ]:
import requests
from custom_token_auth import TokenAuth #custom_token_auth is your custom token authorization

TOKEN = "<YOUR_GITHUB_PA_TOKEN>"

with requests.Session() as session:
    session.auth = TokenAuth(TOKEN)

    first_response = session.get("https://api.github.com/user")
    second_response = session.get("https://api.github.com/user")

print(first_response.headers)
print(second_response.json())

ModuleNotFoundError: No module named 'custom_token_auth'

#### Retry Failed Requests


When a request fails, you might want your application to retry the same request. However, Requests won’t do this for you by default. To apply this functionality, you need to implement a custom transport adapter.

In [38]:
import requests
from requests.adapters import HTTPAdapter
from requests.exceptions import RetryError
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total=2,
    status_forcelist=[429, 500, 502, 503, 504]
)
github_adapter = HTTPAdapter(max_retries=retry_strategy)

with requests.Session() as session:
    session.mount("https://api.github.com", github_adapter)
    try:
        response = session.get("https://api.github.com/")
    except RetryError as err:
        print(f"Error: {err}")